In [60]:
from datetime import datetime
from pathlib import Path
import re
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font

In [88]:
FORMAT_TANGGAL_JAM = "%d-%m-%Y_%H-%M-%S"
tanggal_jam = datetime.now().strftime(FORMAT_TANGGAL_JAM)

In [62]:
folder = Path("../scrap_status_usaha")
files = list(folder.glob("status_usaha*.xlsx"))
if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder scrap_status_usaha")


def extract_timestamp(f):
    # sesuaikan regex ini dengan format timestamp di nama file kamu
    match = re.search(r"(\d{8}_\d{6})", f.stem)  # contoh: 20250706_123045
    return match.group(1) if match else ""


file_terbaru = max(files, key=extract_timestamp)
print("Membaca file:", file_terbaru)
df_usaha = pd.read_excel(file_terbaru, dtype=str)
print(len(df_usaha["id_wilayah"].unique()))
print(len(df_usaha))
df_usaha.head()

Membaca file: ..\scrap_status_usaha\status_usaha_sls_sumsel_20260715_135259.xlsx
40835
612344


,id_wilayah,nama_wilayah,level_wilayah,nama_provinsi,nama_kabupaten,nama_kecamatan,nama_desa,nama_sls,is_agregat,kode_indikator,nama_indikator,satuan,total_value,updated_at,kd_kab
0,1601052001000100,RT 01 DUSUN 1,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 01 DUSUN 1,0,14,Jumlah Keluarga Prelist Awal,Keluarga,63,NaN,1601
1,1601052001000200,RT 02 DUSUN 1,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 02 DUSUN 1,0,14,Jumlah Keluarga Prelist Awal,Keluarga,75,NaN,1601
2,1601052001000300,RT 03 DUSUN 1,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 03 DUSUN 1,0,14,Jumlah Keluarga Prelist Awal,Keluarga,39,NaN,1601
3,1601052001000400,RT 04 DUSUN 2,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 04 DUSUN 2,0,14,Jumlah Keluarga Prelist Awal,Keluarga,38,NaN,1601
4,1601052001000500,RT 05 DUSUN 2,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 05 DUSUN 2,0,14,Jumlah Keluarga Prelist Awal,Keluarga,65,NaN,1601


In [63]:
# 1. pivot seperti biasa TANPA dropna=False (pakai default dropna=True)
df_usaha_pivot = df_usaha.pivot_table(
    index=["kd_kab", "nama_kecamatan", "nama_desa", "id_wilayah", "nama_sls"],
    columns="nama_indikator",
    values="total_value",
    aggfunc="first",
).reset_index()

# 2. daftar kolom indikator yang diinginkan
kolom_indikator = [
    "Jumlah UB Prelist Awal",
    "Jumlah UM Prelist Awal",
    "Jumlah UMK Prelist Awal",
    "Jumlah Usaha Ditemukan (BKU)",
    "Jumlah Usaha Ditutup (BKU)",
    "Jumlah Usaha Ganda (BKU)",
    "Jumlah Usaha Tidak Ditemukan (BKU)",
    "Jumlah Usaha Baru (BKU)",
    "Jumlah Keluarga Prelist Awal",
    "Jumlah Usaha Ditemukan (Usaha Keluarga)",
    "Jumlah Usaha Tutup (Usaha Keluarga)",
    "Jumlah Usaha Ganda (Usaha Keluarga)",
    "Jumlah Usaha Tidak Ditemukan (Usaha Keluarga)",
    "Jumlah Usaha Baru (Usaha Keluarga)",
    "Progres Pendataan Usaha dalam Keluarga",
]

# 3. tambahkan kolom yang hilang (karena semua NaN) secara manual, tanpa cartesian product
kolom_wilayah = ["id_wilayah", "kd_kab", "nama_kecamatan", "nama_desa", "nama_sls"]

for kolom in kolom_indikator:
    if kolom not in df_usaha_pivot.columns:
        df_usaha_pivot[kolom] = pd.NA  # atau 0, sesuai kebutuhan

# 4. urutkan kolom sesuai spec
df_usaha_pivot = df_usaha_pivot[kolom_wilayah + kolom_indikator]

# 5. isi NaN dengan 0 kalau perlu
df_usaha_pivot[kolom_indikator] = df_usaha_pivot[kolom_indikator].fillna(0)

df_usaha_pivot = df_usaha_pivot.sort_values(by="id_wilayah").reset_index(drop=True)
df_usaha_pivot

C:\Users\lenov\AppData\Local\Temp\ipykernel_268232\1323553531.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_usaha_pivot[kolom_indikator] = df_usaha_pivot[kolom_indikator].fillna(0)


nama_indikator,id_wilayah,kd_kab,nama_kecamatan,nama_desa,nama_sls,Jumlah UB Prelist Awal,Jumlah UM Prelist Awal,Jumlah UMK Prelist Awal,Jumlah Usaha Ditemukan (BKU),Jumlah Usaha Ditutup (BKU),Jumlah Usaha Ganda (BKU),Jumlah Usaha Tidak Ditemukan (BKU),Jumlah Usaha Baru (BKU),Jumlah Keluarga Prelist Awal,Jumlah Usaha Ditemukan (Usaha Keluarga),Jumlah Usaha Tutup (Usaha Keluarga),Jumlah Usaha Ganda (Usaha Keluarga),Jumlah Usaha Tidak Ditemukan (Usaha Keluarga),Jumlah Usaha Baru (Usaha Keluarga),Progres Pendataan Usaha dalam Keluarga
0,1601000000000000,1601,TIDAK DIKETAHUI,TIDAK DIKETAHUI,TIDAK DIKETAHUI,13,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1601052001000100,1601,LENGKITI,BANDAR JAYA,RT 01 DUSUN 1,0,0,23,3,0,0,14,0,63,15,0,0,0,15,30
2,1601052001000200,1601,LENGKITI,BANDAR JAYA,RT 02 DUSUN 1,0,0,20,1,0,0,9,0,75,17,0,0,0,12,29
3,1601052001000300,1601,LENGKITI,BANDAR JAYA,RT 03 DUSUN 1,0,0,4,0,0,0,0,1,39,5,0,0,0,1,6
4,1601052001000400,1601,LENGKITI,BANDAR JAYA,RT 04 DUSUN 2,0,0,13,0,0,0,0,0,38,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38318,1674042010000102,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 01,0,0,8,2,0,0,6,3,104,19,0,0,0,29,48
38319,1674042010000200,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 02,0,0,18,0,0,0,2,0,129,0,0,0,0,0,0
38320,1674042010000300,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 03,0,0,9,0,0,0,8,1,63,6,0,0,0,5,11
38321,1674042010000400,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 04,0,0,6,0,0,0,0,0,120,0,0,0,0,0,0


In [64]:
kolom_perlu_numerik = [
    "Jumlah UB Prelist Awal",
    "Jumlah UM Prelist Awal",
    "Jumlah UMK Prelist Awal",
    "Jumlah Usaha Ditemukan (BKU)",
    "Jumlah Usaha Baru (BKU)",
    "Jumlah Usaha Ditemukan (Usaha Keluarga)",
    "Jumlah Usaha Baru (Usaha Keluarga)",
]

for kolom in kolom_perlu_numerik:
    df_usaha_pivot[kolom] = pd.to_numeric(df_usaha_pivot[kolom], errors="coerce")

df_usaha_pivot["jumlah_prelist_usaha"] = df_usaha_pivot[
    [
        "Jumlah UB Prelist Awal",
        "Jumlah UM Prelist Awal",
        "Jumlah UMK Prelist Awal",
    ]
].sum(axis=1)

df_usaha_pivot["jumlah_usaha_realisasi"] = df_usaha_pivot[
    [
        "Jumlah Usaha Ditemukan (BKU)",
        "Jumlah Usaha Baru (BKU)",
        "Jumlah Usaha Ditemukan (Usaha Keluarga)",
        "Jumlah Usaha Baru (Usaha Keluarga)",
    ]
].sum(axis=1)

df_usaha_pivot

nama_indikator,id_wilayah,kd_kab,nama_kecamatan,nama_desa,nama_sls,Jumlah UB Prelist Awal,Jumlah UM Prelist Awal,Jumlah UMK Prelist Awal,Jumlah Usaha Ditemukan (BKU),Jumlah Usaha Ditutup (BKU),...,Jumlah Usaha Baru (BKU),Jumlah Keluarga Prelist Awal,Jumlah Usaha Ditemukan (Usaha Keluarga),Jumlah Usaha Tutup (Usaha Keluarga),Jumlah Usaha Ganda (Usaha Keluarga),Jumlah Usaha Tidak Ditemukan (Usaha Keluarga),Jumlah Usaha Baru (Usaha Keluarga),Progres Pendataan Usaha dalam Keluarga,jumlah_prelist_usaha,jumlah_usaha_realisasi
0,1601000000000000,1601,TIDAK DIKETAHUI,TIDAK DIKETAHUI,TIDAK DIKETAHUI,13,0,0,0,0,...,0,0,0,0,0,0,0,0,13,0
1,1601052001000100,1601,LENGKITI,BANDAR JAYA,RT 01 DUSUN 1,0,0,23,3,0,...,0,63,15,0,0,0,15,30,23,33
2,1601052001000200,1601,LENGKITI,BANDAR JAYA,RT 02 DUSUN 1,0,0,20,1,0,...,0,75,17,0,0,0,12,29,20,30
3,1601052001000300,1601,LENGKITI,BANDAR JAYA,RT 03 DUSUN 1,0,0,4,0,0,...,1,39,5,0,0,0,1,6,4,7
4,1601052001000400,1601,LENGKITI,BANDAR JAYA,RT 04 DUSUN 2,0,0,13,0,0,...,0,38,0,0,0,0,0,0,13,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38318,1674042010000102,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 01,0,0,8,2,0,...,3,104,19,0,0,0,29,48,8,53
38319,1674042010000200,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 02,0,0,18,0,0,...,0,129,0,0,0,0,0,0,18,0
38320,1674042010000300,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 03,0,0,9,0,0,...,1,63,6,0,0,0,5,11,9,12
38321,1674042010000400,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 04,0,0,6,0,0,...,0,120,0,0,0,0,0,0,6,0


In [65]:
folder = Path("../scrap_status_keluarga")
files = list(folder.glob("status_keluarga*.xlsx"))
if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder scrap_status_usaha")


def extract_timestamp(f):
    # sesuaikan regex ini dengan format timestamp di nama file kamu
    match = re.search(r"(\d{8}_\d{6})", f.stem)  # contoh: 20250706_123045
    return match.group(1) if match else ""


file_terbaru = max(files, key=extract_timestamp)
print("Membaca file:", file_terbaru)
df_keluarga = pd.read_excel(file_terbaru, dtype=str)
print(len(df_keluarga["id_wilayah"].unique()))
print(len(df_keluarga))
df_keluarga.head()

Membaca file: ..\scrap_status_keluarga\status_keluarga_sls_sumsel_20260715_141229.xlsx
40837
367488


,id_wilayah,nama_wilayah,level_wilayah,nama_provinsi,nama_kabupaten,nama_kecamatan,nama_desa,nama_sls,is_agregat,kode_indikator,nama_indikator,satuan,total_value,updated_at,kd_kab
0,1601052001000100,RT 01 DUSUN 1,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 01 DUSUN 1,0,14,Jumlah Keluarga Prelist Awal,Keluarga,63,NaN,1601
1,1601052001000200,RT 02 DUSUN 1,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 02 DUSUN 1,0,14,Jumlah Keluarga Prelist Awal,Keluarga,75,NaN,1601
2,1601052001000300,RT 03 DUSUN 1,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 03 DUSUN 1,0,14,Jumlah Keluarga Prelist Awal,Keluarga,39,NaN,1601
3,1601052001000400,RT 04 DUSUN 2,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 04 DUSUN 2,0,14,Jumlah Keluarga Prelist Awal,Keluarga,38,NaN,1601
4,1601052001000500,RT 05 DUSUN 2,sub_sls,SUMATERA SELATAN,OGAN KOMERING ULU,LENGKITI,BANDAR JAYA,RT 05 DUSUN 2,0,14,Jumlah Keluarga Prelist Awal,Keluarga,65,NaN,1601


In [66]:
# 1. pivot seperti biasa TANPA dropna=False (pakai default dropna=True)
df_keluarga_pivot = df_keluarga.pivot_table(
    index=["kd_kab", "nama_kecamatan", "nama_desa", "id_wilayah", "nama_sls"],
    columns="nama_indikator",
    values="total_value",
    aggfunc="first",
).reset_index()

# 2. daftar kolom indikator yang diinginkan
kolom_indikator = [
    "Jumlah Keluarga Prelist Awal",
    "Jumlah Keluarga Ditemukan",
    "Jumlah Keluarga Meninggal",
    "Jumlah Keluarga Tidak Eligible",
    "Jumlah Keluarga Tidak Dapat Ditemui Sampai Akhir Pendataan",
    "Jumlah Keluarga Tidak Ditemukan",
    "Jumlah Keluarga Baru",
    "Jumlah Keluarga Menolak Didata",
    # "Jumlah Keluarga Khusus",
]

# 3. tambahkan kolom yang hilang (karena semua NaN) secara manual, tanpa cartesian product
kolom_wilayah = ["id_wilayah", "kd_kab", "nama_kecamatan", "nama_desa", "nama_sls"]

for kolom in kolom_indikator:
    if kolom not in df_keluarga_pivot.columns:
        df_keluarga_pivot[kolom] = pd.NA  # atau 0, sesuai kebutuhan

# 4. urutkan kolom sesuai spec
df_keluarga_pivot = df_keluarga_pivot[kolom_wilayah + kolom_indikator]

# 5. isi NaN dengan 0 kalau perlu
df_keluarga_pivot[kolom_indikator] = df_keluarga_pivot[kolom_indikator].fillna(0)

df_keluarga_pivot = df_keluarga_pivot.sort_values(by="id_wilayah").reset_index(drop=True)
df_keluarga_pivot

C:\Users\lenov\AppData\Local\Temp\ipykernel_268232\3808109139.py:33: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_keluarga_pivot[kolom_indikator] = df_keluarga_pivot[kolom_indikator].fillna(0)


nama_indikator,id_wilayah,kd_kab,nama_kecamatan,nama_desa,nama_sls,Jumlah Keluarga Prelist Awal,Jumlah Keluarga Ditemukan,Jumlah Keluarga Meninggal,Jumlah Keluarga Tidak Eligible,Jumlah Keluarga Tidak Dapat Ditemui Sampai Akhir Pendataan,Jumlah Keluarga Tidak Ditemukan,Jumlah Keluarga Baru,Jumlah Keluarga Menolak Didata
0,1601000000000000,1601,TIDAK DIKETAHUI,TIDAK DIKETAHUI,TIDAK DIKETAHUI,0,0,0,0,0,0,0,0
1,1601052001000100,1601,LENGKITI,BANDAR JAYA,RT 01 DUSUN 1,63,32,2,0,0,19,10,0
2,1601052001000200,1601,LENGKITI,BANDAR JAYA,RT 02 DUSUN 1,75,35,0,0,0,21,11,0
3,1601052001000300,1601,LENGKITI,BANDAR JAYA,RT 03 DUSUN 1,39,4,0,0,0,9,1,0
4,1601052001000400,1601,LENGKITI,BANDAR JAYA,RT 04 DUSUN 2,38,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
37941,1674042010000102,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 01,104,79,2,0,0,15,18,0
37942,1674042010000200,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 02,129,1,0,0,0,2,0,0
37943,1674042010000300,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 03,63,27,1,0,0,4,5,0
37944,1674042010000400,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 04,120,0,1,0,0,0,0,0


In [67]:
kolom_perlu_numerik = ["Jumlah Keluarga Ditemukan", "Jumlah Keluarga Baru"]

for kolom in kolom_perlu_numerik:
    df_keluarga_pivot[kolom] = pd.to_numeric(df_keluarga_pivot[kolom], errors="coerce")

df_keluarga_pivot["jumlah_keluarga_realisasi"] = df_keluarga_pivot[
    ["Jumlah Keluarga Ditemukan", "Jumlah Keluarga Baru"]
].sum(axis=1)

df_keluarga_pivot

nama_indikator,id_wilayah,kd_kab,nama_kecamatan,nama_desa,nama_sls,Jumlah Keluarga Prelist Awal,Jumlah Keluarga Ditemukan,Jumlah Keluarga Meninggal,Jumlah Keluarga Tidak Eligible,Jumlah Keluarga Tidak Dapat Ditemui Sampai Akhir Pendataan,Jumlah Keluarga Tidak Ditemukan,Jumlah Keluarga Baru,Jumlah Keluarga Menolak Didata,jumlah_keluarga_realisasi
0,1601000000000000,1601,TIDAK DIKETAHUI,TIDAK DIKETAHUI,TIDAK DIKETAHUI,0,0,0,0,0,0,0,0,0
1,1601052001000100,1601,LENGKITI,BANDAR JAYA,RT 01 DUSUN 1,63,32,2,0,0,19,10,0,42
2,1601052001000200,1601,LENGKITI,BANDAR JAYA,RT 02 DUSUN 1,75,35,0,0,0,21,11,0,46
3,1601052001000300,1601,LENGKITI,BANDAR JAYA,RT 03 DUSUN 1,39,4,0,0,0,9,1,0,5
4,1601052001000400,1601,LENGKITI,BANDAR JAYA,RT 04 DUSUN 2,38,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37941,1674042010000102,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 01,104,79,2,0,0,15,18,0,97
37942,1674042010000200,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 02,129,1,0,0,0,2,0,0,1
37943,1674042010000300,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 03,63,27,1,0,0,4,5,0,32
37944,1674042010000400,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 04,120,0,1,0,0,0,0,0,0


In [81]:
df_pivot_merged = df_usaha_pivot.merge(
    df_keluarga_pivot,
    left_on="id_wilayah",  # ganti sesuai nama kolom key di df1
    right_on="id_wilayah",  # ganti sesuai nama kolom key di df2 (kalau namanya beda)
    how="outer",
    indicator=True,  # ini kuncinya
)

In [82]:
df_pivot_merged
df_pivot_merged["nama_sls"] = df_pivot_merged["nama_sls_x"].combine_first(
    df_pivot_merged["nama_sls_y"]
)
df_pivot_merged["kd_kab"] = df_pivot_merged["kd_kab_x"].combine_first(
    df_pivot_merged["kd_kab_y"]
)
df_pivot_merged

nama_indikator,id_wilayah,kd_kab_x,nama_kecamatan_x,nama_desa_x,nama_sls_x,Jumlah UB Prelist Awal,Jumlah UM Prelist Awal,Jumlah UMK Prelist Awal,Jumlah Usaha Ditemukan (BKU),Jumlah Usaha Ditutup (BKU),...,Jumlah Keluarga Meninggal,Jumlah Keluarga Tidak Eligible,Jumlah Keluarga Tidak Dapat Ditemui Sampai Akhir Pendataan,Jumlah Keluarga Tidak Ditemukan,Jumlah Keluarga Baru,Jumlah Keluarga Menolak Didata,jumlah_keluarga_realisasi,_merge,nama_sls,kd_kab
0,1601000000000000,1601,TIDAK DIKETAHUI,TIDAK DIKETAHUI,TIDAK DIKETAHUI,13.0,0.0,0.0,0.0,0,...,0,0,0.0,0,0.0,0.0,0.0,both,TIDAK DIKETAHUI,1601
1,1601052001000100,1601,LENGKITI,BANDAR JAYA,RT 01 DUSUN 1,0.0,0.0,23.0,3.0,0,...,2,0,0.0,19,10.0,0.0,42.0,both,RT 01 DUSUN 1,1601
2,1601052001000200,1601,LENGKITI,BANDAR JAYA,RT 02 DUSUN 1,0.0,0.0,20.0,1.0,0,...,0,0,0.0,21,11.0,0.0,46.0,both,RT 02 DUSUN 1,1601
3,1601052001000300,1601,LENGKITI,BANDAR JAYA,RT 03 DUSUN 1,0.0,0.0,4.0,0.0,0,...,0,0,0.0,9,1.0,0.0,5.0,both,RT 03 DUSUN 1,1601
4,1601052001000400,1601,LENGKITI,BANDAR JAYA,RT 04 DUSUN 2,0.0,0.0,13.0,0.0,0,...,0,0,0.0,0,0.0,0.0,0.0,both,RT 04 DUSUN 2,1601
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38388,1674042010000102,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 01,0.0,0.0,8.0,2.0,0,...,2,0,0.0,15,18.0,0.0,97.0,both,RT 01,1674
38389,1674042010000200,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 02,0.0,0.0,18.0,0.0,0,...,0,0,0.0,2,0.0,0.0,1.0,both,RT 02,1674
38390,1674042010000300,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 03,0.0,0.0,9.0,0.0,0,...,1,0,0.0,4,5.0,0.0,32.0,both,RT 03,1674
38391,1674042010000400,1674,LUBUK LINGGAU UTARA II,KALI SERAYU,RT 04,0.0,0.0,6.0,0.0,0,...,1,0,0.0,0,0.0,0.0,0.0,both,RT 04,1674


In [84]:
df_pivot_merged = df_pivot_merged[
    [
        "id_wilayah",
        "kd_kab",
        "nama_sls",
        "Jumlah UB Prelist Awal",
        "Jumlah UM Prelist Awal",
        "Jumlah UMK Prelist Awal",
        "Jumlah Usaha Ditemukan (BKU)",
        "Jumlah Usaha Ditutup (BKU)",
        "Jumlah Usaha Ganda (BKU)",
        "Jumlah Usaha Tidak Ditemukan (BKU)",
        "Jumlah Usaha Baru (BKU)",
        "Jumlah Usaha Ditemukan (Usaha Keluarga)",
        "Jumlah Usaha Tutup (Usaha Keluarga)",
        "Jumlah Usaha Ganda (Usaha Keluarga)",
        "Jumlah Usaha Tidak Ditemukan (Usaha Keluarga)",
        "Jumlah Usaha Baru (Usaha Keluarga)",
        "Progres Pendataan Usaha dalam Keluarga",
        "Jumlah Keluarga Ditemukan",
        "Jumlah Keluarga Meninggal",
        "Jumlah Keluarga Tidak Eligible",
        "Jumlah Keluarga Tidak Dapat Ditemui Sampai Akhir Pendataan",
        "Jumlah Keluarga Tidak Ditemukan",
        "Jumlah Keluarga Baru",
        "Jumlah Keluarga Menolak Didata",
        "jumlah_prelist_usaha",
        "jumlah_usaha_realisasi",
        "Jumlah Keluarga Prelist Awal_x",
        "jumlah_keluarga_realisasi",
    ]
].rename(
    columns={
        "Jumlah Keluarga Prelist Awal_x":"jumlah_prelist_keluarga"
    }
)
df_pivot_merged

nama_indikator,id_wilayah,kd_kab,nama_sls,Jumlah UB Prelist Awal,Jumlah UM Prelist Awal,Jumlah UMK Prelist Awal,Jumlah Usaha Ditemukan (BKU),Jumlah Usaha Ditutup (BKU),Jumlah Usaha Ganda (BKU),Jumlah Usaha Tidak Ditemukan (BKU),...,Jumlah Keluarga Meninggal,Jumlah Keluarga Tidak Eligible,Jumlah Keluarga Tidak Dapat Ditemui Sampai Akhir Pendataan,Jumlah Keluarga Tidak Ditemukan,Jumlah Keluarga Baru,Jumlah Keluarga Menolak Didata,jumlah_prelist_usaha,jumlah_usaha_realisasi,jumlah_prelist_keluarga,jumlah_keluarga_realisasi
0,1601000000000000,1601,TIDAK DIKETAHUI,13.0,0.0,0.0,0.0,0,0,0,...,0,0,0.0,0,0.0,0.0,13.0,0.0,0,0.0
1,1601052001000100,1601,RT 01 DUSUN 1,0.0,0.0,23.0,3.0,0,0,14,...,2,0,0.0,19,10.0,0.0,23.0,33.0,63,42.0
2,1601052001000200,1601,RT 02 DUSUN 1,0.0,0.0,20.0,1.0,0,0,9,...,0,0,0.0,21,11.0,0.0,20.0,30.0,75,46.0
3,1601052001000300,1601,RT 03 DUSUN 1,0.0,0.0,4.0,0.0,0,0,0,...,0,0,0.0,9,1.0,0.0,4.0,7.0,39,5.0
4,1601052001000400,1601,RT 04 DUSUN 2,0.0,0.0,13.0,0.0,0,0,0,...,0,0,0.0,0,0.0,0.0,13.0,0.0,38,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38388,1674042010000102,1674,RT 01,0.0,0.0,8.0,2.0,0,0,6,...,2,0,0.0,15,18.0,0.0,8.0,53.0,104,97.0
38389,1674042010000200,1674,RT 02,0.0,0.0,18.0,0.0,0,0,2,...,0,0,0.0,2,0.0,0.0,18.0,0.0,129,1.0
38390,1674042010000300,1674,RT 03,0.0,0.0,9.0,0.0,0,0,8,...,1,0,0.0,4,5.0,0.0,9.0,12.0,63,32.0
38391,1674042010000400,1674,RT 04,0.0,0.0,6.0,0.0,0,0,0,...,1,0,0.0,0,0.0,0.0,6.0,0.0,120,0.0


In [85]:
df_daftar_ppl = pd.read_excel("../20260608 Rekap Prelist_16_ppl_pml.xlsx", dtype=str)
df_daftar_ppl = df_daftar_ppl.dropna(subset=["IDSUBSLS_25_2"]).reset_index(drop=True)
df_daftar_ppl


,IDKAB,IDKEC,IDDESA,IDSUBSLS_25_2,PPL,PML,KDPROV,NMPROV,KDKAB,NMKAB,...,ASSIGNMENT KELUARGA DAN ROSTER USAHA DI DALAM KELUARGA,Unnamed: 25,Unnamed: 26,ASSIGNMENT BKU & BANGUNAN LAINNYA,Unnamed: 28,Unnamed: 29,Unnamed: 30,TOTAL ASSIGNMENT FASIH,Flag SLS Open PBI,KK Open PBI
0,1601,1601052,1601052001,1601052001000100,blackphiton123@gmail.com,cecepbta09@gmail.com,16,SUMATERA SELATAN,01,OGAN KOMERING ULU,...,63,0,54,23,0,0,0,86,0,0
1,1601,1601052,1601052001,1601052001000200,blackphiton123@gmail.com,cecepbta09@gmail.com,16,SUMATERA SELATAN,01,OGAN KOMERING ULU,...,75,0,59,20,0,0,0,95,0,0
2,1601,1601052,1601052001,1601052001000300,blackphiton123@gmail.com,cecepbta09@gmail.com,16,SUMATERA SELATAN,01,OGAN KOMERING ULU,...,39,0,23,4,0,0,0,43,0,0
3,1601,1601052,1601052001,1601052001000400,blackphiton123@gmail.com,cecepbta09@gmail.com,16,SUMATERA SELATAN,01,OGAN KOMERING ULU,...,38,0,15,13,0,0,0,51,0,0
4,1601,1601052,1601052001,1601052001000500,blackphiton123@gmail.com,cecepbta09@gmail.com,16,SUMATERA SELATAN,01,OGAN KOMERING ULU,...,65,0,38,4,0,0,0,69,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40816,1674,1674042,1674042010,1674042010000200,dwipuspita1423@gmail.com,whycm93@gmail.com,16,SUMATERA SELATAN,74,LUBUKLINGGAU,...,129,46,12,18,0,0,5,152,0,0
40817,1674,1674042,1674042010,1674042010000300,dwipuspita1423@gmail.com,whycm93@gmail.com,16,SUMATERA SELATAN,74,LUBUKLINGGAU,...,63,18,3,9,0,0,0,72,0,0
40818,1674,1674042,1674042010,1674042010000400,dwipuspita1423@gmail.com,whycm93@gmail.com,16,SUMATERA SELATAN,74,LUBUKLINGGAU,...,121,57,16,6,0,0,2,129,0,0
40819,1674,1674042,1674042010,1674042010200100,dwipuspita1423@gmail.com,whycm93@gmail.com,16,SUMATERA SELATAN,74,LUBUKLINGGAU,...,0,0,0,0,0,0,0,0,0,0


In [92]:
df_pivot_merged_2 = df_pivot_merged.merge(
    df_daftar_ppl[["IDSUBSLS_25_2", "PPL", "PML"]],
    left_on="id_wilayah",
    right_on="IDSUBSLS_25_2",
    how="left",
).drop(columns={"IDSUBSLS_25_2"})

In [93]:
df_pivot_merged_2

,id_wilayah,kd_kab,nama_sls,Jumlah UB Prelist Awal,Jumlah UM Prelist Awal,Jumlah UMK Prelist Awal,Jumlah Usaha Ditemukan (BKU),Jumlah Usaha Ditutup (BKU),Jumlah Usaha Ganda (BKU),Jumlah Usaha Tidak Ditemukan (BKU),...,Jumlah Keluarga Tidak Dapat Ditemui Sampai Akhir Pendataan,Jumlah Keluarga Tidak Ditemukan,Jumlah Keluarga Baru,Jumlah Keluarga Menolak Didata,jumlah_prelist_usaha,jumlah_usaha_realisasi,jumlah_prelist_keluarga,jumlah_keluarga_realisasi,PPL,PML
0,1601000000000000,1601,TIDAK DIKETAHUI,13.0,0.0,0.0,0.0,0,0,0,...,0.0,0,0.0,0.0,13.0,0.0,0,0.0,NaN,NaN
1,1601052001000100,1601,RT 01 DUSUN 1,0.0,0.0,23.0,3.0,0,0,14,...,0.0,19,10.0,0.0,23.0,33.0,63,42.0,blackphiton123@gmail.com,cecepbta09@gmail.com
2,1601052001000200,1601,RT 02 DUSUN 1,0.0,0.0,20.0,1.0,0,0,9,...,0.0,21,11.0,0.0,20.0,30.0,75,46.0,blackphiton123@gmail.com,cecepbta09@gmail.com
3,1601052001000300,1601,RT 03 DUSUN 1,0.0,0.0,4.0,0.0,0,0,0,...,0.0,9,1.0,0.0,4.0,7.0,39,5.0,blackphiton123@gmail.com,cecepbta09@gmail.com
4,1601052001000400,1601,RT 04 DUSUN 2,0.0,0.0,13.0,0.0,0,0,0,...,0.0,0,0.0,0.0,13.0,0.0,38,0.0,blackphiton123@gmail.com,cecepbta09@gmail.com
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38388,1674042010000102,1674,RT 01,0.0,0.0,8.0,2.0,0,0,6,...,0.0,15,18.0,0.0,8.0,53.0,104,97.0,dwipuspita1423@gmail.com,whycm93@gmail.com
38389,1674042010000200,1674,RT 02,0.0,0.0,18.0,0.0,0,0,2,...,0.0,2,0.0,0.0,18.0,0.0,129,1.0,dwipuspita1423@gmail.com,whycm93@gmail.com
38390,1674042010000300,1674,RT 03,0.0,0.0,9.0,0.0,0,0,8,...,0.0,4,5.0,0.0,9.0,12.0,63,32.0,dwipuspita1423@gmail.com,whycm93@gmail.com
38391,1674042010000400,1674,RT 04,0.0,0.0,6.0,0.0,0,0,0,...,0.0,0,0.0,0.0,6.0,0.0,120,0.0,dwipuspita1423@gmail.com,whycm93@gmail.com


In [90]:
import os

folder_output = "../Rekap Progres Pendataan"
os.makedirs(folder_output, exist_ok=True)

df_pivot_merged_2.to_excel(
    f"{folder_output}/rekap_progres_pendataan_{tanggal_jam}.xlsx", index=False
)

In [95]:
import os
import sqlite3

folder_db = "../SQLLITE"
os.makedirs(folder_db, exist_ok=True)

conn = sqlite3.connect(f"{folder_db}/rekap_progress_pendataan.db")

df_pivot_merged_2.to_sql(
    "rekap_progress_pendataan", conn, if_exists="replace", index=False
)

cursor = conn.cursor()
cursor.execute("""
    CREATE UNIQUE INDEX IF NOT EXISTS idx_id_wilayah 
    ON rekap_progress_pendataan (id_wilayah)
""")
conn.commit()
conn.close()

In [100]:
def pastikan_kolom_ada(df, db_path, table):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # cek apakah tabel sudah ada
    cursor.execute(
        f"SELECT name FROM sqlite_master WHERE type='table' AND name=?", (table,)
    )
    tabel_ada = cursor.fetchone()

    if tabel_ada:
        # ambil daftar kolom yang sudah ada di tabel
        cursor.execute(f'PRAGMA table_info("{table}")')
        kolom_ada = [row[1] for row in cursor.fetchall()]

        # bandingkan dengan kolom di df, tambahkan kolom yang belum ada
        for kolom in df.columns:
            if kolom not in kolom_ada:
                cursor.execute(f'ALTER TABLE "{table}" ADD COLUMN "{kolom}" TEXT')
                print(f"Kolom '{kolom}' ditambahkan ke tabel {table}")

        conn.commit()

    conn.close()


def upsert_ke_sqlite(
    df,
    tanggal_jam,
    db_path="../SQLLITE/rekap_progress_pendataan.db",
    table="rekap_progress_pendataan",
):
    df = df.copy()
    df["last_update"] = tanggal_jam

    # pastikan semua kolom di df tersedia di tabel
    pastikan_kolom_ada(df, db_path, table)

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    kolom = df.columns.tolist()
    placeholder = ", ".join(["?"] * len(kolom))
    kolom_str = ", ".join([f'"{k}"' for k in kolom])

    data = df.where(pd.notnull(df), None).values.tolist()

    cursor.executemany(
        f'INSERT OR REPLACE INTO "{table}" ({kolom_str}) VALUES ({placeholder})', data
    )

    conn.commit()
    conn.close()
    print(f"{len(df)} baris berhasil di-upsert ke {table} pada {tanggal_jam}")

In [101]:
upsert_ke_sqlite(df_pivot_merged_2, tanggal_jam)

Kolom 'last_update' ditambahkan ke tabel rekap_progress_pendataan
38393 baris berhasil di-upsert ke rekap_progress_pendataan pada 15-07-2026_15-33-02
